In [ ]:
#importações
import asyncio
from typing import List, Optional
import httpx
from bs4 import BeautifulSoup
import json
from fastAPI import FastAPI, Query
from pydantic import BaseModel, HttUrl

In [ ]:
#Pydantic valida os dados que entram e saem da API
class PrecoProduto(BaseModel):
    loja:str
    titulo:str
    preco:float
    link:HttUrl
    em_estoque:bool
    imagem_url:Optional[str] = None

class RespotaBusca(BaseModel):
    termo:str
    resultado: List[PrecoProduto]
    tempo_execucao:float

async def estrair_dados_json_ld(html_content: str, url: str, nome_loja: str) -> Optional[PrecoProduto]:
    '''
    Função auxiliar para varrer o HTML em busca  de dados  estruturados (JSON-LD).
    Isso evita quebras constantes por mudança de layout visual.
    '''
    try:
        soup = BeatifulSoup(html_content, "html.parser")
        #Procura por scripts do tipo application/ld+json
        scripts = soup.find_all("script", type="application/ld+json")

        for script in scripts:
            try:
                data = json.loads(script.string)